# 02 — Feature Exploration

Build persistence diagrams from transit graphs, inspect individual diagrams,
and construct the feature matrix used for clustering.

**Prerequisites:** Run `gtfs-features --scale agglomeration` first to produce `.npz` diagrams.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt

from swiss_gtfs.features.persistence import load_diagrams, diagram_path
from swiss_gtfs.features.vectorize import (
    vectorize_city_stats,
    vectorize_city_landscape,
    build_feature_matrix,
)

In [ ]:
# --- Configuration ---
SCALE        = 'agglomeration'
DIAGRAMS_DIR = '../outputs/diagrams'
CITY         = 'zurich'

In [ ]:
# Inspect a single city's persistence diagrams
npz_path = diagram_path(CITY, SCALE, DIAGRAMS_DIR)
diagrams = load_diagrams(npz_path)

print(f'H0 bars: {len(diagrams["h0"])}')
print(f'H1 bars: {len(diagrams["h1"])}')

In [ ]:
# Plot the H0 persistence diagram
h0 = diagrams['h0']
finite_h0 = h0[~np.isinf(h0[:, 1])]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].scatter(finite_h0[:, 0], finite_h0[:, 1], s=8, alpha=0.6)
axes[0].set_title(f'H0 diagram — {CITY}')
axes[0].set_xlabel('Birth')
axes[0].set_ylabel('Death')
diag = [0, finite_h0.max() if len(finite_h0) else 1]
axes[0].plot(diag, diag, 'k--', alpha=0.3)

h1 = diagrams['h1']
if len(h1):
    finite_h1 = h1[~np.isinf(h1[:, 1])]
    axes[1].scatter(finite_h1[:, 0], finite_h1[:, 1], s=8, alpha=0.6, color='orange')
axes[1].set_title(f'H1 diagram — {CITY}')
axes[1].set_xlabel('Birth')
axes[1].set_ylabel('Death')

plt.tight_layout()
plt.show()

In [ ]:
# Build the full feature matrix for the scale
keys, matrix = build_feature_matrix(
    scale=SCALE,
    diagrams_dir=DIAGRAMS_DIR,
    method='stats',
)
print(f'Feature matrix: {matrix.shape} ({len(keys)} cities × {matrix.shape[1]} features)')

In [ ]:
# Quick feature-space overview
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

scaled = StandardScaler().fit_transform(matrix)
pca = PCA(n_components=2)
coords = pca.fit_transform(scaled)

df_pca = pd.DataFrame({'city': keys, 'pc1': coords[:, 0], 'pc2': coords[:, 1]})

plt.figure(figsize=(8, 6))
plt.scatter(df_pca['pc1'], df_pca['pc2'], s=30)
for _, row in df_pca.iterrows():
    plt.annotate(row['city'], (row['pc1'], row['pc2']), fontsize=6, alpha=0.7)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.title(f'PCA of persistence features — {SCALE}')
plt.tight_layout()
plt.show()